# Pivotal-Token Steering on Qwen3-0.6B — GSM8K Evaluation

Interactive, checkpointed port of the steering + evaluation pipeline from the `Pivotal-Token-Representation-Learning` repo. Designed to run unchanged on:

- **Google Colab** (free GPU works well; CPU works but is slow)
- **Kaggle / SageMaker Studio / RunPod** (any Jupyter kernel)

### What it does

1. Loads Qwen3-0.6B from Hugging Face — never caches the model into the repo.
2. Loads a pre-computed **pivotal-token steering vector** (centroid difference at layer 14, the best-performing probe layer with AUROC 0.808 on the probing task).
3. Runs a paired GSM8K evaluation of the base model vs. the same model with the steering vector added at that layer with factors `{1.2, 1.4, 1.6, 1.8, 2.0, 0.0, -1.0}`.
   * `factor = 1 + α`, where `α` is the coefficient added to the pivotal direction.
   * `factor = 1.2` → mild amplification; `factor = 2.0` → strong amplification; `factor = 0` → ablation of the pivotal direction; `factor = -1` → anti-steering.
4. Reports **accuracy**, **binary F1** (correct-vs-not), and **answer parse-rate**.
5. Produces error-analysis plots and saves all intermediate artifacts to `./nb_results/`.

Each heavy cell caches its output to disk, so re-running the notebook only regenerates what's missing (set `FORCE_RERUN = True` in the config cell to redo everything).

## 1. Install dependencies

In [ ]:
# Run once per runtime. Safe to skip if already installed.
%pip install -q "transformers>=4.44" datasets accelerate matplotlib seaborn scikit-learn tqdm pandas

## 2. Imports, seeding, device detection

In [ ]:
import json
import os
import random
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

RESULTS_DIR = Path("nb_results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"device={DEVICE} dtype={DTYPE}")
print(f"results -> {RESULTS_DIR.resolve()}")

## 3. Experiment configuration

Edit here if you want to sweep a different layer, factor set, or example count. All downstream cells reference these globals.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
LAYER = 14  # top-performing probe layer (AUROC 0.808 on the probing task)
FACTORS = [1.2, 1.4, 1.6, 1.8, 2.0, 0.0, -1.0]
MAX_EXAMPLES = 100
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.6
TOP_P = 0.9

# Where to fetch the pre-computed steering vector. Fallback order:
#   1. LOCAL_PROBE_DIR (repo is cloned)
#   2. PROBE_RAW_URL_BASE + filename (GitHub raw download)
#   3. Manual upload (uncomment the cell below the probe loader)
LOCAL_PROBE_DIR = Path("artifacts/cached3/sklearn/steering_configs")
PROBE_RAW_URL_BASE = (
    "https://raw.githubusercontent.com/"
    "stvngo/Pivotal-Token-Representation-Learning/"
    "main/artifacts/cached3/sklearn/steering_configs/"
)

FORCE_RERUN = False  # set True to ignore cached results in nb_results/

## 4. Load the pre-computed steering probe

In [ ]:
import urllib.request


def _try_local_probe(layer: int):
    cfg_path = LOCAL_PROBE_DIR / f"steering_layer{layer}.json"
    vec_path = LOCAL_PROBE_DIR / f"steering_layer{layer}_vector.npy"
    if cfg_path.exists() and vec_path.exists():
        return cfg_path, vec_path
    return None, None


def _try_remote_probe(layer: int, dest_dir: Path):
    dest_dir.mkdir(parents=True, exist_ok=True)
    cfg_name = f"steering_layer{layer}.json"
    vec_name = f"steering_layer{layer}_vector.npy"
    try:
        urllib.request.urlretrieve(PROBE_RAW_URL_BASE + cfg_name, dest_dir / cfg_name)
        urllib.request.urlretrieve(PROBE_RAW_URL_BASE + vec_name, dest_dir / vec_name)
        return dest_dir / cfg_name, dest_dir / vec_name
    except Exception as exc:
        print(f"Remote fetch failed: {exc}")
        return None, None


_probe_dir = RESULTS_DIR / "probes"
cfg_path, vec_path = _try_local_probe(LAYER)
if cfg_path is None:
    cfg_path, vec_path = _try_remote_probe(LAYER, _probe_dir)

assert cfg_path is not None and vec_path is not None, (
    "Could not locate steering probe. Either clone the repo so LOCAL_PROBE_DIR exists, "
    "set PROBE_RAW_URL_BASE to a reachable GitHub raw URL, or uncomment the manual-upload cell below."
)

with cfg_path.open("r") as f:
    steering_cfg = json.load(f)
steering_vector = np.load(vec_path).astype(np.float32)

print(f"Loaded probe config from {cfg_path}")
print(f"  layer       = {steering_cfg['layer']}")
print(f"  vector_type = {steering_cfg['vector_type']}")
print(f"  hidden_dim  = {steering_cfg['hidden_dim']}")
print(f"  vector_norm = {steering_cfg['vector_norm']:.4f}")
assert steering_cfg["hidden_dim"] == steering_vector.shape[0]

In [ ]:
# --- Google Colab: upload your own probe --------------------------------
# Uncomment the three lines below to upload a `steering_layer{N}.json` and
# matching `steering_layer{N}_vector.npy` from your local machine. The files
# land in the current working directory; set LOCAL_PROBE_DIR = Path('.') in the
# config cell above and re-run the probe loader.
#
# from google.colab import files
# uploaded = files.upload()
# print('Uploaded:', list(uploaded.keys()))

## 5. Load Qwen3-0.6B from Hugging Face

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

_t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME} on {DEVICE} ({DTYPE}) in {time.time() - _t0:.1f}s")
print(f"  params            = {n_params / 1e9:.3f}B")
print(f"  hidden_size       = {model.config.hidden_size}")
print(f"  num_hidden_layers = {model.config.num_hidden_layers}")
assert steering_vector.shape[0] == model.config.hidden_size, (
    f"Probe hidden_dim {steering_vector.shape[0]} ≠ model hidden_size {model.config.hidden_size}"
)

## 6. Steering hooks

Ported verbatim from `probe_pipeline/steering.py` so the notebook is self-contained. We register a forward hook on the layer's *output*; the hook adds `strength * v` to every token's hidden state, where `strength = factor - 1.0`.

In [ ]:
import torch.nn as nn


def _get_decoder_layer(mdl: nn.Module, layer_idx: int) -> nn.Module:
    if hasattr(mdl, "model") and hasattr(mdl.model, "layers"):
        return mdl.model.layers[layer_idx]
    if hasattr(mdl, "transformer") and hasattr(mdl.transformer, "h"):
        return mdl.transformer.h[layer_idx]
    if hasattr(mdl, "decoder") and hasattr(mdl.decoder, "layers"):
        return mdl.decoder.layers[layer_idx]
    raise AttributeError(f"Cannot find decoder layers in {type(mdl)}")


def _make_steering_hook(vector: torch.Tensor, strength: float):
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        v = vector.to(hidden.device).to(hidden.dtype)
        if v.dim() == 1:
            v = v.view(1, 1, -1)
        delta = strength * v
        if isinstance(output, tuple):
            return (hidden + delta,) + output[1:]
        return hidden + delta

    return hook


def register_steering(mdl, layer: int, vector: torch.Tensor, strength: float):
    return [
        _get_decoder_layer(mdl, layer).register_forward_hook(
            _make_steering_hook(vector, strength)
        )
    ]


def remove_hooks(handles):
    for h in handles:
        h.remove()


_vec_tensor = torch.from_numpy(steering_vector).to(torch.float32)
print(f"Steering vector: shape={tuple(_vec_tensor.shape)} norm={_vec_tensor.norm().item():.4f}")

## 7. Answer parser + metrics

Same logic used by the CLI (`probe_pipeline/gsm8k_eval.py`) and covered by `tests/test_answer_extraction.py`. The inline assertions re-validate it before we commit to a long eval run.

In [ ]:
import re


def extract_gsm8k_answer(text: str):
    m = re.search(r"####\s*([+-]?\d+(?:,\d{3})*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else None


def is_correct(pred, gt):
    if pred is None or gt is None:
        return False
    p = pred.strip().replace(",", "")
    g = gt.strip().replace(",", "")
    try:
        return float(p) == float(g)
    except ValueError:
        return p == g


def compute_metrics(results):
    n = len(results)
    if n == 0:
        return {"accuracy": 0.0, "f1": 0.0, "parse_rate": 0.0, "correct": 0, "n": 0}
    correct = sum(1 for r in results if r["correct"])
    parsed = sum(1 for r in results if r["predicted"] is not None)
    # Binary F1 with positive class = "got the right answer".
    # TP = correct; FP = parsed-but-wrong; FN = not correct (includes unparsed).
    tp = correct
    fp = parsed - correct
    fn = n - correct
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "accuracy": correct / n,
        "f1": f1,
        "parse_rate": parsed / n,
        "correct": correct,
        "n": n,
    }


# Inline parser sanity check — mirrors tests/test_answer_extraction.py.
assert extract_gsm8k_answer("#### 42") == "42"
assert extract_gsm8k_answer("He has #### 1,234 apples.") == "1234"
assert extract_gsm8k_answer("The net change is #### -7") == "-7"
assert extract_gsm8k_answer("So the answer is #### 3.50") == "3.50"
assert extract_gsm8k_answer("...so the total is 18 apples.") == "18"
assert extract_gsm8k_answer("") is None
assert is_correct("3.0", "3") is True
assert is_correct("1234", "1,234") is True
assert is_correct(None, "5") is False
print("Parser sanity checks passed.")

## 8. GSM8K test subset (seeded)

In [ ]:
from datasets import load_dataset

_ds_full = load_dataset("openai/gsm8k", "main", split="test")
_rng = np.random.default_rng(SEED)
_indices = sorted(
    _rng.choice(len(_ds_full), size=min(MAX_EXAMPLES, len(_ds_full)), replace=False).tolist()
)
ds_subset = _ds_full.select(_indices)
print(f"GSM8K test subset: {len(ds_subset)} examples (seed={SEED})")

examples = []
for row in ds_subset:
    examples.append({
        "question": row["question"],
        "ground_truth": extract_gsm8k_answer(row["answer"]),
        "answer_full": row["answer"],
    })

print("\n--- preview ---")
for ex in examples[:2]:
    print("Q :", ex["question"][:160].replace("\n", " "))
    print("GT:", ex["ground_truth"])
    print("---")

## 9. Generation helper + base evaluation

Re-seeds the RNG at the start of each run so base and steered generations are paired as closely as Hugging Face generation allows.

In [ ]:
def generate_once(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def _reseed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


def evaluate(label: str, strength):
    _reseed()
    handles = []
    if strength is not None:
        handles = register_steering(model, LAYER, _vec_tensor, strength)
    results = []
    try:
        for i, ex in enumerate(tqdm(examples, desc=label)):
            prompt = f"Question: {ex['question']}\n\nLet's think step by step.\n\n"
            text = generate_once(prompt)
            pred = extract_gsm8k_answer(text)
            ok = is_correct(pred, ex["ground_truth"])
            results.append({
                "idx": i,
                "question": ex["question"],
                "ground_truth": ex["ground_truth"],
                "predicted": pred,
                "correct": ok,
                "full_output": text[-1000:],
            })
    finally:
        remove_hooks(handles)
    return results, compute_metrics(results)


def save_run(label: str, results, metrics, extra_state=None):
    (RESULTS_DIR / f"{label}_results.json").write_text(json.dumps(results, indent=2))
    (RESULTS_DIR / f"{label}_generations.txt").write_text(
        "\n\n====\n\n".join(
            f"[{r['idx']}] gt={r['ground_truth']} pred={r['predicted']} correct={r['correct']}\n{r['full_output']}"
            for r in results
        )
    )
    state = {
        "label": label,
        "model": MODEL_NAME,
        "layer": LAYER,
        "seed": SEED,
        "max_examples": MAX_EXAMPLES,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "device": DEVICE,
        "metrics": metrics,
    }
    if extra_state:
        state.update(extra_state)
    (RESULTS_DIR / f"{label}_state.json").write_text(json.dumps(state, indent=2))


base_cache = RESULTS_DIR / "base_results.json"
if base_cache.exists() and not FORCE_RERUN:
    base_results = json.loads(base_cache.read_text())
    base_metrics = compute_metrics(base_results)
    print("Loaded cached base results.")
else:
    base_results, base_metrics = evaluate("base", strength=None)
    save_run("base", base_results, base_metrics)
print(
    f"base: acc={base_metrics['accuracy']:.3f} "
    f"f1={base_metrics['f1']:.3f} "
    f"parse={base_metrics['parse_rate']:.3f}"
)

## 10. Steered sweep

Each factor produces its own `*_results.json`, `*_generations.txt`, and `*_state.json` inside `nb_results/`. The state file is the "saved new state" of each steered variant — a pure configuration artifact (layer, factor, strength, vector reference, seed) since hooks are applied at runtime rather than mutating weights.

In [ ]:
all_results = {"base": base_results}
all_metrics = {"base": base_metrics}

for factor in FACTORS:
    label = f"steered_{factor}"
    strength = factor - 1.0
    cache = RESULTS_DIR / f"{label}_results.json"
    if cache.exists() and not FORCE_RERUN:
        res = json.loads(cache.read_text())
        met = compute_metrics(res)
        print(f"[cached] {label}: acc={met['accuracy']:.3f} f1={met['f1']:.3f} parse={met['parse_rate']:.3f}")
    else:
        res, met = evaluate(label, strength=strength)
        save_run(label, res, met, extra_state={
            "factor": factor,
            "strength": strength,
            "vector_type": steering_cfg["vector_type"],
            "vector_norm": steering_cfg["vector_norm"],
            "vector_path": str(vec_path.name),
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        })
        print(f"[done]   {label}: acc={met['accuracy']:.3f} f1={met['f1']:.3f} parse={met['parse_rate']:.3f}")
    all_results[label] = res
    all_metrics[label] = met

## 11. Summary table

In [ ]:
import pandas as pd

rows = []
for name, m in all_metrics.items():
    rows.append({
        "model": name,
        "correct": m["correct"],
        "n": m["n"],
        "accuracy": round(m["accuracy"], 4),
        "f1": round(m["f1"], 4),
        "parse_rate": round(m["parse_rate"], 4),
    })
summary_df = pd.DataFrame(rows)
summary_df.to_csv(RESULTS_DIR / "summaries.csv", index=False)
(RESULTS_DIR / "summaries.json").write_text(json.dumps(all_metrics, indent=2))
summary_df

## 12. Error analysis + visualizations

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110

sorted_factors = sorted(FACTORS)
acc = [all_metrics[f"steered_{f}"]["accuracy"] for f in sorted_factors]
f1s = [all_metrics[f"steered_{f}"]["f1"] for f in sorted_factors]
pr = [all_metrics[f"steered_{f}"]["parse_rate"] for f in sorted_factors]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sorted_factors, acc, marker="o", label="Steered accuracy")
ax.plot(sorted_factors, f1s, marker="s", label="Steered F1")
ax.axhline(base_metrics["accuracy"], ls="--", color="gray", alpha=0.7, label="Base accuracy")
ax.axhline(base_metrics["f1"], ls=":", color="gray", alpha=0.7, label="Base F1")
ax.set_xlabel("Steering factor")
ax.set_ylabel("Score")
ax.set_title(f"GSM8K accuracy / F1 vs steering factor (layer {LAYER})")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "accuracy_f1_vs_factor.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.5))
labels_pr = [str(f) for f in sorted_factors] + ["base"]
values_pr = pr + [base_metrics["parse_rate"]]
colors = ["#4c72b0"] * len(sorted_factors) + ["#888"]
ax.bar(labels_pr, values_pr, color=colors)
ax.set_ylim(0, 1.02)
ax.set_ylabel("Parse rate")
ax.set_xlabel("Factor")
ax.set_title("Fraction of generations with an extractable answer")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "parse_rate.png")
plt.show()

# Gained / lost vs base (paired by index).
base_by_idx = {r["idx"]: r["correct"] for r in base_results}
gained, lost, labels = [], [], []
flip_table = []
for f in sorted_factors:
    name = f"steered_{f}"
    res = all_results[name]
    g = sum(1 for r in res if not base_by_idx.get(r["idx"], False) and r["correct"])
    l = sum(1 for r in res if base_by_idx.get(r["idx"], False) and not r["correct"])
    gained.append(g)
    lost.append(l)
    labels.append(str(f))
    for r in res:
        was = base_by_idx.get(r["idx"], False)
        if was != r["correct"]:
            flip_table.append({
                "factor": f,
                "idx": r["idx"],
                "base_correct": was,
                "steered_correct": r["correct"],
                "ground_truth": r["ground_truth"],
                "steered_pred": r["predicted"],
            })

x = np.arange(len(labels))
w = 0.38
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(x - w / 2, gained, w, label="Base wrong → steered right", color="seagreen", alpha=0.85)
ax.bar(x + w / 2, lost, w, label="Base right → steered wrong", color="indianred", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Steering factor")
ax.set_ylabel("Count")
ax.set_title("Error flips vs base (paired on example index)")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "gained_lost.png")
plt.show()

(RESULTS_DIR / "confusion_flips.json").write_text(json.dumps(flip_table, indent=2))
print(f"Saved {len(flip_table)} flip records to confusion_flips.json")

## 13. Bundle everything and (optionally) download

In [ ]:
zip_path = Path("nb_results.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(RESULTS_DIR.parent))
print(f"Zipped {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files  # type: ignore

    files.download(str(zip_path))
except Exception:
    print("Not on Colab — the zip is on the local filesystem.")